# CHELSA-TraCE21k: monthly GeoTIFFs → yearly netCDF

## What this script does

It takes the 2652 monthly GeoTIFF files from CHELSA-TraCE21k and produces a
single netCDF file containing one **yearly mean** raster per 100-year time step.
By default it also **downsamples spatially** to match the TraCE21k T31 grid
(~3.75°), keeping the output file small and directly comparable to the
underlying climate model.

Input: 221 time steps × 12 months = **2652 GeoTIFFs**, each ~100 MB on disk
(~3.6 GB unpacked in RAM), storing daily maximum air temperature (`tasmax`)
as Kelvin × 10.

Output (default, with coarsening on): **1 netCDF file**, ~3 MB, 221 time
slices × 96 longitudes × 46 latitudes, in proper Kelvin.

Output (coarsening off): **1 netCDF file**, ~250 GB, 221 time slices × 43200
longitudes × 20880 latitudes.

## How it works (3 steps)

### Step 1: Group files by year

Each filename looks like `CHELSA_TraCE21k_tasmax_<MM>_<TTT>_V.1.0.tif`,
where `MM` is the month (`01`–`12`) and `TTT` is a CHELSA time index.
The script reads the time index from the filename and converts it to a
calendar year using a single linear formula:

```
year = 1990 − (20 − TTT) × 100
```

Mapping:

| TTT     | Year (CE) | Comment             |
|---------|-----------|---------------------|
| `0020`  | 1990      | most recent step    |
| `0000`  | -10       | ~ year 0            |
| `-001`  | -110      | 100 years older     |
| `-200`  | -20010    | ~ 22 ka BP, ≈ LGM   |

After this step, every year has exactly 12 file paths attached to it.
The script aborts if any year has fewer or more than 12 months — this catches
incomplete downloads early instead of producing silently wrong yearly means.

### Step 2: Build a yearly mean per time step

For each year, the script opens its 12 monthly files **lazily** with
`rioxarray` and dask chunks. Lazy loading means the rasters are not actually
read into memory until needed — essential because each file unpacks to ~3.6 GB
in RAM.

If coarsening is enabled (default), each monthly array is downsampled by a
block-mean of 450 × 450 pixels before further processing. This shrinks the
arrays by a factor of ~200,000, which makes everything that follows almost
free (RAM and CPU).

The 12 monthly arrays are then stacked along a new `month` axis and averaged.
The result is multiplied by `0.1` to undo CHELSA's scale factor, giving real
Kelvin values. A timestamp (`<year>-07-01`) is attached as the `time`
coordinate.

### Step 3: Combine and save

All 221 yearly slices are concatenated along the `time` dimension and written
to a single compressed netCDF file with the variable named `tasmax`.

## Progress reporting

The script reports progress in two phases, because dask is lazy: building
the computation plan (fast) and actually executing it (slow).

**Phase 1 — Building graph** (tqdm bar): goes through all 221 time steps.
This finishes in seconds because no pixel data is read yet — only metadata
and the dask graph are built.

```
Building graph: 100%|████████████| 221/221 [00:08<00:00, 26.5it/s]
```

**Phase 2 — Writing netCDF** (dask ProgressBar): this is where the real
work happens. The bar shows the share of total dask tasks completed.

```
[                                        ] | 0% Completed | 12.3 s
[######                                  ] | 16% Completed | 4 min 12 s
[################                        ] | 41% Completed | 11 min 5 s
[########################################] | 100% Completed | 28 min 7 s
```

If the second phase looks stuck at 0% for a long time, that is normal —
dask first inspects every input file before starting the actual reads.

## Spatial coarsening (downsampling)

The script applies a block-mean reduction with factor **450** in both x and y:

| Dimension | Native | Coarsened | Factor |
|-----------|--------|-----------|--------|
| longitude | 43200  | 96        | 450    |
| latitude  | 20880  | 46        | 450    |
| pixel size | 30 arc-sec (~1 km) | ~3.75° | 450 |

Why 450? Because the TraCE21k climate model is computed on the T31 spectral
grid (96 × 48). Coarsening CHELSA to 96 × 46 places it on essentially the
same grid, so the comparison no longer requires a separate regridding step
later. (CHELSA only covers latitudes from -90° to +84°, hence 46 rows
instead of T31's full 48.)

Why coarsen at all? The native CHELSA resolution is ~1 km, but TraCE21k is
~400 km. Any comparison will end up on the coarser grid, so keeping the
fine detail is a waste of disk and processing time.

### Disabling the coarsening

Open the script and locate this block near the top:

```python
COARSEN_FACTOR = 450
```

Set it to `None`:

```python
COARSEN_FACTOR = None
```

That deactivates the coarsening step entirely and keeps the full ~1 km
resolution. **Only do this if you have at least 300 GB of free disk space
and 32+ GB of RAM** — you do not.

## ⚠ Verify the time mapping before trusting the output

The mapping `year = 1990 − (20 − TTT) × 100` is a hypothesis. CHELSA's
official documentation does not explicitly publish the index-to-year table.
The hypothesis is consistent with:

- the metadata field `frequency=centennial` in your `.tif` files,
- the paper's claim of ~21,000 years of coverage,
- the file count (221 timesteps × 100 years ≈ 22 ka).

But before processing all 2652 files, do this 5-minute sanity check:

1. Open `CHELSA_TraCE21k_tasmax_07_0020_V.1.0.tif` (July of t=20, the
   hypothesised 1990 CE) in QGIS.
2. Sample a pixel for Berlin. Multiply the raw value by `0.1` and subtract
   `273.15` — this gives July 1990 max temperature in °C. It should be
   roughly 24 °C. If you get a value near 0 °C or below, t=20 is **not**
   1990 — it is probably the LGM, and the time axis must be inverted.
3. Repeat for `CHELSA_TraCE21k_tasmax_07_-200_V.1.0.tif`. This should be
   distinctly colder (LGM Europe was 5–15 K colder than today, depending on
   location).

If steps 2 and 3 give inverted results, change the formula to
`year = -20010 + (TTT + 200) × 100` (i.e. anchor t = -200 at 22 ka BP
instead of anchoring t = 20 at 1990).

## Important notes

- **Units**: output is in **Kelvin**. Subtract 273.15 to get Celsius.
- **Time calendar**: the script uses `cftime` because pandas cannot represent
  years before 1677 — and CHELSA-TraCE21k goes back to ~20,000 BCE.
- **Memory**: with coarsening on, peak RAM stays well under 1 GB. With
  coarsening off, dask chunking still keeps it under ~1 GB but the dask
  graph becomes very large (~265,000 tasks).
- **Runtime**: 1–3 hours on a modern laptop. The bottleneck is reading and
  decompressing the 2652 input GeoTIFFs, not the actual computation.

## Required Python packages

```bash
conda install -c conda-forge xarray rioxarray cftime netcdf4 dask tqdm
```

In [1]:
"""
Convert CHELSA-TraCE21k monthly tasmax GeoTIFFs into one netCDF file
holding yearly means at 100-year time steps.

NOTE: The time-index → calendar-year mapping is HYPOTHESIZED, not officially
documented. Verify with a sample file before trusting the time axis (see README).
"""

import re
from pathlib import Path

import cftime
import rioxarray
import xarray as xr
from dask.diagnostics import ProgressBar
from tqdm import tqdm


# --- Settings ---------------------------------------------------------------

variable = 'tasmax'
#variable = 'tasmin'

INPUT_DIR = Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'data'
OUTPUT_NC = Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'output' / f'{variable}.nc'
OUTPUT_NC.parent.mkdir(parents=True, exist_ok=True)

# Filename pattern: CHELSA_TraCE21k_tasmax_<MM>_<TTT>_V.1.0.tif
FILENAME = re.compile(rf"CHELSA_TraCE21k_{variable}_(\d{{2}})_(-?\d+)_V\.1\.0\.tif")

# Chunks aligned with COARSEN_FACTOR (4500 = 10×450, 2250 = 5×450).
# This keeps each coarsen block inside a single chunk → less I/O, less RAM.
CHUNKS = {"x": 4500, "y": 2250}


# ============================================================================
#                  ↓↓↓   COARSENING — DOWNSAMPLING BLOCK   ↓↓↓
# ----------------------------------------------------------------------------
# Reduces spatial resolution from CHELSA's native 30 arc-sec (~1 km) down to
# ~3.75°, which matches the TraCE21k climate-model grid (T31, 96 × 48).
#
# Factor 450:
#     43200 / 450 = 96 longitudes
#     20880 / 450 = 46 latitudes
#
# Effect on output size: from several hundred GB → a few MB.
#
# >>> TO DISABLE on a machine with more RAM/disk: set COARSEN_FACTOR = None.
# >>> That keeps the full ~1 km resolution. WARNING: with coarsening off,
# >>> the eager .compute() below will OOM. Switch to per-year file output.
# ============================================================================
COARSEN_FACTOR = 450
# ============================================================================


# --- Helper -----------------------------------------------------------------

def index_to_year(ti: int) -> int:
    """
    Convert CHELSA time index to calendar year.
    All timesteps are centennial; anchor: t=20 -> 1990 CE.

      t =   20 -> 1990 CE
      t =    0 ->  -10 CE
      t = -200 -> -20010 CE  (~22 ka BP)
    """
    return 1990 - (20 - ti) * 100


# --- Step 1: group files by year --------------------------------------------

files_by_year = {}
for path in INPUT_DIR.glob(f"CHELSA_TraCE21k_{variable}_*.tif"):
    match = FILENAME.match(path.name)
    if not match:
        continue
    month = int(match.group(1))
    year = index_to_year(int(match.group(2)))
    files_by_year.setdefault(year, []).append((month, path))

if not files_by_year:
    raise SystemExit(f"No matching .tif files found in {INPUT_DIR}")

print(f"Processing {len(files_by_year)} time steps:")


# --- Step 2: build a yearly mean for each time step -------------------------
# After coarsening, one yearly slice is ~96×46 pixels (~18 KB), so we
# .compute() each year immediately. This drops the big dask chunks from memory before moving on to the next year.

yearly_slices = []
for year in tqdm(sorted(files_by_year), desc="Computing yearly means"):
    monthly_files = sorted(files_by_year[year])

    # sanity check: every year must have all 12 months
    if len(monthly_files) != 12:
        raise ValueError(
            f"Year {year} has {len(monthly_files)} months, expected 12"
        )

    monthly_arrays = []
    for _, path in monthly_files:
        da = rioxarray.open_rasterio(path, chunks=CHUNKS, masked=True).squeeze("band", drop=True)

        # >>> COARSENING applied here — see banner block above <<<
        if COARSEN_FACTOR:
            da = da.coarsen(
                x=COARSEN_FACTOR, y=COARSEN_FACTOR, boundary="trim"
            ).mean()

        monthly_arrays.append(da)

    yearly_mean = xr.concat(monthly_arrays, dim="month").mean(dim="month")
    yearly_mean = (yearly_mean * 0.1).compute()  # CHELSA scale factor: Kelvin * 10
                                                 # .compute() forces evaluation
                                                 # NOW so the big chunks are freed

    timestamp = cftime.DatetimeProlepticGregorian(year, 7, 1)
    yearly_slices.append(yearly_mean.expand_dims(time=[timestamp]))


# --- Step 3: combine and save -----------------------------------------------

combined = xr.concat(yearly_slices, dim="time").rename({"x": "lon", "y": "lat"})
combined.name = variable
combined.attrs = {
    "long_name": "Yearly mean of daily maximum near-surface air temperature",
    "units": "K",
    "source": "CHELSA-TraCE21k v1.0 (centennial)",
    "comment": "Each timestep is a centennial climatology averaged over 12 months.",
}
if COARSEN_FACTOR:
    combined.attrs["spatial_processing"] = (
        f"Spatially downsampled by factor {COARSEN_FACTOR} (block-mean) "
        f"to approximately match TraCE21k T31 grid (~3.75°)."
    )

print(f"Writing {OUTPUT_NC}")
with ProgressBar():
    combined.to_netcdf(
        OUTPUT_NC,
        encoding={variable: {"zlib": True, "complevel": 4, "dtype": "float32"}},
        unlimited_dims=["time"],
    )
print("Done.")

Processing 221 time steps:


Computing yearly means: 100%|██████████████████████████████████████████████████████████| 221/221 [2:10:53<00:00, 35.54s/it]

Writing /home/lghomeoffice/git/paleoclimate_model_comparison/data/CHELSA-TraCE21k-centennial/output/tasmax.nc
Done.


In [2]:
# REPEAT FOR TASMIN THEN COMPUTE TASMEAN !!!

## PalMod2: Yearly Mean Computation & Merge

Computes annual means from monthly NetCDF files and merges them into a single output file using **CDO** (Climate Data Operators).

**Steps:**
1. Collect all `.nc` files from `INPUT_DIR`.
2. For each file: apply `cdo yearmean` (collapses 12 monthly values → 1 yearly mean) and save to a temporary directory.
3. Merge all yearly-mean files along the time axis with `cdo mergetime` into a single output file.

**Notes:**
- Temporary files are auto-deleted after the cell finishes.
- `tqdm` shows progress for the per-file step (the slow part).
- Time axis runs forward in model years (oldest → youngest), as the simulation integrates forward from the Last Glacial Maximum.

In [3]:
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory
from tqdm import tqdm

# repeat for PMMXMCRTDGr111Amtasgn30201_1-250 and PMMXMCRTDGr111Lmtslgn30201_1-250!

# --- settings ---
INPUT_DIR = Path.cwd() / 'data' / 'PalMod2' / 'data' / 'PMMXMCRTDGr111Lmtslgn30201_1-250'
OUTPUT_NC = Path.cwd() / 'data' / 'PalMod2' / 'output' / 'PMMXMCRTDGr111Lmtslgn30201_1-250.nc'
OUTPUT_NC.parent.mkdir(parents=True, exist_ok=True)

# --- collect input files ---
nc_files = sorted(INPUT_DIR.glob("*.nc"))
print(f"Found {len(nc_files)} files.")

# --- process ---
with TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    yearmean_files = []

    # Step 1: yearly mean for each file
    for f in tqdm(nc_files, desc="Computing yearly means"):
        out = tmp / f"{f.stem}_ymean.nc"
        subprocess.run(["cdo", "yearmean", str(f), str(out)], check=True)
        yearmean_files.append(str(out))

    # Step 2: merge all into one file
    print("Merging...")
    subprocess.run(["cdo", "mergetime", *yearmean_files, str(OUTPUT_NC)], check=True)

print(f"Done: {OUTPUT_NC}")

Found 250 files.


Computing yearly means:   0%|                                                                      | 0/250 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'cdo'